# 🧠 MedQCNN: Hybrid Quantum-Classical CNN for Medical Diagnostics

## What This Notebook Covers

This notebook is your comprehensive guide to the **MedQCNN** project — a hybrid quantum-classical neural network designed for medical image diagnostics. By the end, you'll understand:

1. **Why** quantum computing helps with medical image classification
2. **How** the classical CNN compresses images into quantum-compatible vectors
3. **How** the quantum circuit processes data in Hilbert space
4. **How** to train and run the complete hybrid pipeline

---

## 1. The Problem: Why Quantum for Medical Imaging?

### The Classical Crisis

Classical deep learning models (ResNet, VGG, etc.) work well for image classification, but they face critical challenges in **medical imaging**:

| Problem | Impact |
|---------|--------|
| **Parameter Bloat** | ResNet-50 has ~25M parameters; way too many for small medical datasets |
| **Data Scarcity** | Clinical MRI datasets may have only 100–1000 samples (vs. millions in ImageNet) |
| **Overfitting** | Too many parameters + too little data = memorization, not generalization |
| **High Dimensionality** | 3D MRI volumes can be 256×256×128 = millions of voxels |

### The Quantum Advantage

A quantum circuit with **n qubits** can represent $2^n$ dimensional states simultaneously via **superposition**. With just 8 qubits, we access a **256-dimensional** Hilbert space $\mathcal{H}_{2^8}$ using only ~64 trainable parameters (vs. thousands in classical FC layers).

**Key insight**: The quantum circuit evaluates highly non-linear decision boundaries using *exponentially fewer parameters* than classical networks.

## 2. Architecture Overview

MedQCNN is a **two-node pipeline**:

```
Medical Image → [Node A: Classical CNN] → z ∈ R^256 → [Node B: Quantum Circuit] → ⟨σ_z⟩ → Diagnosis
```

- **Node A (Classical)**: Frozen ResNet-18 backbone + FC projector → L2-normalized vector $\mathbf{z} \in \mathbb{R}^{256}$
- **Node B (Quantum)**: Amplitude encoding + variational ansatz + Pauli-Z measurement → expectation values

Let's explore each component by building them step by step.

In [8]:
# ===== SETUP: Ensure medqcnn is importable =====
import sys
from pathlib import Path

# Add the project root (parent of notebooks/) to Python path
project_root = str(Path.cwd().parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    print(f"Added to path: {project_root}")

# Now import everything
import torch
import numpy as np
import matplotlib.pyplot as plt
import pennylane as qml

from medqcnn.config.constants import DEMO_QUBITS, NUM_ANSATZ_LAYERS
from medqcnn.utils.device import set_seed

set_seed(42)

# We'll use 4 qubits for fast interactive demos
N_QUBITS = DEMO_QUBITS  # 4
LATENT_DIM = 2 ** N_QUBITS  # 16
print(f"Qubits: {N_QUBITS}, Latent Dim: {LATENT_DIM}")

Added to path: /mnt/6C6E8F7D6E8F3EB8/company/MedQCNN
Qubits: 4, Latent Dim: 16


## 3. Node A: Classical CNN Compression (Phase 1)

### Why Compress?

A medical image might be 224×224×3 = **150,528 pixels**. We can't encode that directly into a quantum circuit — we'd need ~17 qubits (2^17 = 131,072), which is too expensive to simulate.

Instead, we use a **frozen pre-trained ResNet** as a feature extractor to compress the image into a compact vector, then a **fully-connected projector** maps it to exactly $2^n$ dimensions.

### The Pipeline

```
Image (224×224×3) → ResNet-18 (frozen) → feature_dim=512 → FC → R^16 → L2 normalize → ||z||₂ = 1
```

The **L2 normalization** is critical: the vector $\mathbf{z}$ must satisfy $\|\mathbf{z}\|_2 = 1$ because quantum state amplitudes must form a valid probability distribution ($\sum |z_i|^2 = 1$).

In [ ]:
# Demonstrate the classical compression pipeline
from medqcnn.classical.backbone import ClassicalBackbone
from medqcnn.classical.projector import LatentProjector

# Build backbone + projector
backbone = ClassicalBackbone(pretrained=False, freeze=True)
projector = LatentProjector(input_dim=backbone.feature_dim, latent_dim=LATENT_DIM)

# Simulate a batch of 2 grayscale medical images
fake_images = torch.randn(2, 1, 224, 224)

# Step 1: Extract features
with torch.no_grad():
    features = backbone(fake_images)
print(f"Backbone output: {list(features.shape)}")
print(f"  → {features.shape[1]} dimensional feature vector")

# Step 2: Project to quantum-compatible latent space
with torch.no_grad():
    z = projector(features)
print(f"\nProjector output: {list(z.shape)}")
print(f"  → {z.shape[1]} dimensions (= 2^{N_QUBITS})")

# Verify L2 normalization
norms = torch.norm(z, p=2, dim=-1)
print(f"\nL2 norms: {norms.tolist()}")
print(f"  → All ≈ 1.0 ✓ (valid quantum amplitudes)")

## 4. Node B: Quantum Circuit (Phase 2)

This is the heart of MedQCNN. The quantum circuit has 3 stages:

### Stage 1: Amplitude Encoding

We map the classical vector $\mathbf{z} \in \mathbb{R}^{2^n}$ into a quantum state:

$$|\psi(\mathbf{z})\rangle = \sum_{i=0}^{2^n - 1} z_i |i\rangle$$

For 4 qubits, this creates a superposition over 16 basis states:
$|\psi\rangle = z_0|0000\rangle + z_1|0001\rangle + z_2|0010\rangle + \cdots + z_{15}|1111\rangle$

**Why amplitude encoding?** It encodes $2^n$ classical values using only $n$ qubits — an *exponential data compression*.

### Stage 2: Variational Ansatz (Hardware-Efficient)

After encoding, we apply parameterized quantum gates that the optimizer can tune:

- **$R_y(\theta)$** rotation: rotates qubit around Y-axis
- **$R_z(\theta)$** rotation: rotates qubit around Z-axis  
- **$CZ$** gate: creates entanglement between nearest-neighbor qubits (ring topology)

Each layer applies $R_y + R_z$ to every qubit, then $CZ$ gates link them. We repeat for `NUM_ANSATZ_LAYERS` layers.

### Stage 3: Measurement (Local Pauli-Z)

We measure each qubit's expectation value $\langle\sigma_z^{(i)}\rangle \in [-1, 1]$.

**Critical design choice**: We use *local* observables (one per qubit) instead of a global cost function. This is the **barren plateau mitigation** strategy — global cost functions cause gradient variance to vanish exponentially: $\text{Var}(\partial_\theta \mathcal{L}) \propto 2^{-n}$.

In [ ]:
# Let's visualize the quantum circuit!
from medqcnn.quantum.qnode import create_qnode
from medqcnn.quantum.ansatz import get_param_shape
from pennylane import numpy as pnp

# Create a QNode with 4 qubits
qnode = create_qnode(n_qubits=N_QUBITS, n_layers=2)  # 2 layers for visibility

# Create sample inputs
sample_features = np.random.rand(LATENT_DIM)
sample_features = sample_features / np.linalg.norm(sample_features)  # L2 normalize
sample_params = pnp.random.randn(2, N_QUBITS, 2) * 0.1

# Draw the circuit
print("Quantum Circuit Architecture:")
print("=" * 60)
fig, ax = qml.draw_mpl(qnode)(sample_features, sample_params)
plt.tight_layout()
plt.show()

# Run it and see the output
result = qnode(sample_features, sample_params)
print(f"\nCircuit output (averaged ⟨σ_z⟩): {float(result):.6f}")
print(f"  → This value ∈ [-1, 1] represents a quantum 'score'")

In [ ]:
# Count the quantum parameters
param_shape = get_param_shape(n_layers=NUM_ANSATZ_LAYERS, n_qubits=N_QUBITS)
total_quantum_params = np.prod(param_shape)

print(f"Parameter shape: {param_shape}")
print(f"  = {param_shape[0]} layers × {param_shape[1]} qubits × {param_shape[2]} gates")
print(f"  = {total_quantum_params} total quantum parameters")
print(f"\nCompare to a classical FC layer:")
print(f"  16 → 16 FC = {16 * 16 + 16} = 272 params")
print(f"  Quantum saves {272 - total_quantum_params} parameters ({(1 - total_quantum_params/272)*100:.0f}% reduction!)")

## 5. The Barren Plateau Problem

### What Is It?

When training quantum circuits, there's a fundamental problem: for deep circuits with many qubits, the **gradient of the loss function vanishes exponentially**:

$$\text{Var}\left(\frac{\partial \mathcal{L}}{\partial \theta_k}\right) \propto 2^{-n}$$

This means the optimizer sees an essentially flat landscape — it can't find which direction to move.

### How MedQCNN Avoids It

Three strategies from the architecture:

1. **Local observables**: We measure each qubit individually ($\sigma_z^{(i)}$) instead of a global fidelity. Local cost functions have polynomial (not exponential) gradient decay.

2. **Shallow ansatz**: We use only 4 layers (not 20+). Deeper circuits create more barren plateaus.

3. **Small qubit count**: 8 qubits max. The exponential vanishing $2^{-8} = 0.004$ is still manageable; $2^{-20} = 0.000001$ would not be.

## 6. The Complete Hybrid Model

Now let's assemble the full `HybridQCNN` and run an end-to-end forward pass.

In [ ]:
from medqcnn.model.hybrid import HybridQCNN

# Build the model with 4 qubits (demo mode)
model = HybridQCNN(
    n_qubits=N_QUBITS,
    n_layers=NUM_ANSATZ_LAYERS,
    n_classes=2,  # binary: benign vs malignant
    pretrained=False,  # skip ImageNet download for demo
)

# Count parameters
param_counts = model.count_trainable_params()
print("Trainable Parameters by Component")
print("=" * 40)
for name, count in param_counts.items():
    bar = "█" * max(1, count // 5000)
    print(f"  {name:>12s}: {count:>8,}  {bar}")

# Show how few quantum params we need
q_ratio = param_counts['quantum'] / param_counts['total'] * 100
print(f"\n  Quantum params are only {q_ratio:.2f}% of total!")
print(f"  Yet they operate in a {LATENT_DIM}-dimensional Hilbert space.")

In [ ]:
# End-to-end forward pass
fake_batch = torch.randn(2, 1, 224, 224)  # 2 grayscale images

model.eval()
with torch.no_grad():
    logits = model(fake_batch)
    probs = torch.softmax(logits, dim=1)

print("Forward Pass Results")
print("=" * 40)
print(f"  Input:       {list(fake_batch.shape)} (B, C, H, W)")
print(f"  Logits:      {logits.tolist()}")
print(f"  Softmax:     {probs.tolist()}")
print(f"  Predictions: {logits.argmax(dim=1).tolist()}")
print(f"  (0=benign, 1=malignant)")

## 7. Training on Real Medical Data

Let's train the model on **BreastMNIST** — a real breast ultrasound dataset for binary classification (benign vs malignant).

We'll use the 4-qubit demo model for speed.

In [ ]:
# Load BreastMNIST dataset
from medqcnn.data.loader import get_medmnist_loaders

train_loader, val_loader, test_loader = get_medmnist_loaders(
    dataset_name="breastmnist",
    batch_size=8,
    download=True,
)

print(f"Dataset: BreastMNIST (breast ultrasound)")
print(f"  Train: {len(train_loader.dataset)} samples")
print(f"  Val:   {len(val_loader.dataset)} samples")
print(f"  Test:  {len(test_loader.dataset)} samples")

# Visualize some samples
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i in range(4):
    axes[i].imshow(images[i].squeeze(), cmap='gray')
    label_text = "Malignant" if labels[i].item() == 1 else "Benign"
    axes[i].set_title(f"{label_text} ({labels[i].item()})")
    axes[i].axis('off')
plt.suptitle("BreastMNIST Samples", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Build a fresh model for training
model = HybridQCNN(
    n_qubits=N_QUBITS,
    n_layers=NUM_ANSATZ_LAYERS,
    n_classes=2,
    pretrained=False,
)

# Resize wrapper for MedMNIST (28x28) → ResNet (224x224)
class ResizedLoader:
    def __init__(self, loader):
        self.loader = loader
        self.dataset = loader.dataset
    def __iter__(self):
        for imgs, lbls in self.loader:
            imgs = torch.nn.functional.interpolate(
                imgs.float(), size=(224, 224), mode='bilinear', align_corners=False
            )
            yield imgs, lbls
    def __len__(self):
        return len(self.loader)

# Train for 5 epochs (quick demo)
from medqcnn.training.trainer import Trainer

trainer = Trainer(
    model=model,
    train_loader=ResizedLoader(train_loader),
    val_loader=ResizedLoader(val_loader),
    learning_rate=1e-3,
    device='cpu',
)

print("Training for 5 epochs...")
print("(This may take a few minutes on CPU)\n")
history = trainer.fit(epochs=5, save_every=5)

In [ ]:
# Visualize training curves
from medqcnn.training.visualization import plot_training_history

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss', markersize=6)
axes[0].plot(epochs_range, history['val_loss'], 'r-o', label='Val Loss', markersize=6)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['train_acc'], 'b-o', label='Train Acc', markersize=6)
axes[1].plot(epochs_range, history['val_acc'], 'r-o', label='Val Acc', markersize=6)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1.05)

plt.suptitle('MedQCNN Training Progress', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Final Train Acc: {history['train_acc'][-1]:.4f}")
print(f"Final Val Acc:   {history['val_acc'][-1]:.4f}")

## 8. Key Technical Concepts Summary

| Concept | What It Means | Where in Code |
|---------|--------------|---------------|
| **Amplitude Encoding** | Map classical vector to quantum state $\|\psi(z)\rangle$ | `quantum/encoding.py` |
| **Variational Ansatz (HEA)** | Trainable quantum gates ($R_y$, $R_z$, $CZ$) | `quantum/ansatz.py` |
| **Parameter-Shift Rule** | Quantum gradient computation: $\partial f / \partial\theta = [f(\theta+\pi/2) - f(\theta-\pi/2)] / 2$ | PennyLane auto |
| **Barren Plateaus** | Gradient vanishing in quantum circuits ($2^{-n}$) | Mitigated by local observables |
| **TorchLayer** | PennyLane ↔ PyTorch autograd bridge | `quantum/qnode.py` |
| **L2 Normalization** | $\|\mathbf{z}\|_2 = 1$ for valid quantum amplitudes | `classical/projector.py` |
| **Frozen Backbone** | ResNet weights locked — only projector + quantum train | `classical/backbone.py` |
| **Local Observables** | Per-qubit $\langle\sigma_z\rangle$ instead of global cost | `quantum/observables.py` |

### Hardware Constraints

- **8 qubits max** → 256-dim latent space (edge device limit)
- **4 ansatz layers** → shallow to avoid barren plateaus
- **CPU-only simulation** → state-vector simulation on Raspberry Pi 5

### What's Next? (Sprint 3 → 4)

- **Full Training**: Train on 8 qubits with larger MedMNIST datasets
- **Benchmarking**: Compare AUC-ROC against purely classical baselines
- **CaaS-Q Deployment**: Containerize as a Docker microservice with Kafka + LangChain agent

---

## 📚 References

1. **Hybrid Quantum-Classical Neural Networks for Medical Image Classification (2025/2026)**
2. **Quantum Machine Learning Approaches for Medical Image Analysis (2026)**
3. **Mitigating Barren Plateaus in Variational Quantum Algorithms**
4. **Quantum Neural Networks in Magnetic Resonance Imaging**

See `GEMINI.md` for the full project roadmap and architectural spec.